# 🌐 NPPE-1
### Fine-tuning Gemma 3-1B-IT with QLoRA for 13 Indic Languages

**Strategy:** 4-bit quantised Gemma 3 1B + LoRA adapters, trained on all available data,  
with few-shot prompting and majority-vote inference.

# Environment Setup

* Setting `CUDA_VISIBLE_DEVICES="0"` pins everything to a single T4 GPU. This is required because 4-bit quantization + PEFT on multi-GPU setups with `device_map="auto"` tends to break — forcing a single device avoids those issues.
* The `os.walk` loop over `/kaggle/input` is a sanity check to confirm what dataset paths and model paths are visible to this notebook.

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 — required for 4-bit + PEFT

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/adapter_model.safetensors
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/adapter_config.json
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/README.md
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/tokenizer.json
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/tokenizer_config.json
/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter/chat_template.jinja
/kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
/kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/config.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1/model.safetensors
/kaggle/input/m

# Installing libraries

* `transformers` and `accelerate` — for loading the base Gemma 3 model and handling device placement.
* `bitsandbytes` — provides the 4-bit NF4 quantization kernels that let a 1B model fit comfortably in T4 VRAM with headroom for training.
* `peft` — Parameter-Efficient Fine-Tuning. Provides the `LoraConfig` and `get_peft_model` wrappers used to attach LoRA adapters.
* `trl` — gives us `SFTTrainer` and `SFTConfig`, which simplify supervised fine-tuning with features like completion-only loss masking.
* All installed with `-q --upgrade` to keep the output clean and force the latest versions (Gemma 3 support is recent and requires up-to-date libraries).

In [3]:
!pip install -q --upgrade transformers accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 41.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-ad

#  Importing libraries

* `torch`, `numpy`, `pandas` — standard deep-learning and data-handling stack.
* `transformers.BitsAndBytesConfig` — the config object that tells HuggingFace how to quantize the model weights to 4-bit.
* `peft.LoraConfig`, `get_peft_model`, `prepare_model_for_kbit_training` — for attaching LoRA adapters on top of a quantized base model.
* `peft.PeftModel` — used later to load a previously saved LoRA adapter back onto the base model.
* `datasets.Dataset` — HuggingFace's in-memory dataset format; `SFTTrainer` consumes this directly.
* `trl.SFTTrainer`, `SFTConfig` — the high-level training wrapper used to train the LoRA adapters.
* `sklearn.metrics.f1_score`, `classification_report` — for evaluating Macro-F1 (the competition metric) and a per-class breakdown.
* A small `track_time` helper is defined to print elapsed time after each major step — useful for budgeting the 9-hour Kaggle session.

In [4]:
import time
import torch
import numpy as np
import pandas as pd
from peft import PeftModel
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from trl import SFTTrainer, SFTConfig
import bitsandbytes as bnb

def track_time(start, label):
    print(f"{label} — {time.time() - start:.1f}s\n")

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


#  Loading the dataset

* Reading the train and test CSVs. Train has 900 labeled examples across 13 Indic languages; test has 100 unlabeled examples.
* Printing `label.value_counts()` and `language.value_counts()` to sanity-check:
  - Class balance (Positive vs Negative)
  - Per-language sample distribution — with only ~70 samples per language on average, every sample matters.
* `display(train_df.head(3))` gives a visual confirmation of the four columns (`ID`, `sentence`, `label`, `language`). 

In [5]:
t = time.time()

DATA_DIR = "/kaggle/input/competitions/nppe-dlp-2026-term-1"
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(train_df['label'].value_counts())
print(train_df['language'].value_counts())
display(train_df.head(3))

track_time(t, "Data loaded")

Train: (900, 4), Test: (100, 3)
label
Positive    456
Negative    444
Name: count, dtype: int64
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64


,ID,sentence,label,language
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml


Data loaded — 0.1s



# Prompt Design

* The choice of prompt format was arrived at after trial and error with Gemma's chat template.
* `LANG_MAP` converts the ISO language codes in the dataset (e.g. `ta`, `bn`, `ur`) to full language names (`Tamil`, `Bengali`, `Urdu`) so the instruction reads naturally to the model.
* **Few-shot examples** — four hand-picked sentences (Tamil Positive, Bengali Negative, Urdu Positive, Telugu Negative) are included in every prompt. This gives the model concrete anchors for the output format and covers multiple scripts, which matters because the model has to generalize across 13 languages from only ~70 samples each.
* **Chat template** uses Gemma's native `<start_of_turn>user` / `<start_of_turn>model` / `<end_of_turn>` tags. Using the native format is important because the instruct-tuned model was trained with these exact tokens — any other format causes the model to underperform.
* `RESPONSE_TEMPLATE = "<start_of_turn>model\n"` is saved separately because it will be used later to locate where the model's response starts for completion-only loss masking.
* The same `make_prompt` function handles both training (with `label`) and inference (without `label`) — at inference time it stops right after the response template, and the model completes with "Positive" or "Negative".

In [6]:
LANG_MAP = {
    "as": "Assamese", "bd": "Bodo",      "bn": "Bengali",
    "gu": "Gujarati",  "hi": "Hindi",     "kn": "Kannada",
    "ml": "Malayalam", "mr": "Marathi",   "or": "Odia",
    "pa": "Punjabi",   "ta": "Tamil",     "te": "Telugu",
    "ur": "Urdu"
}

RESPONSE_TEMPLATE = "<start_of_turn>model\n"

FEW_SHOT = """Examples:
- "வாடிக்கையாளர் சேவை மிகவும் நன்றாக இருந்தது" → Positive (Tamil)
- "এই পণ্যটি একদম বাজে" → Negative (Bengali)  
- "بہت اچھی سروس ہے" → Positive (Urdu)
- "ఈ సేవ చాలా చెత్తగా ఉంది" → Negative (Telugu)

"""

def make_prompt(sentence: str, language: str, label: str = None) -> str:
    lang_name = LANG_MAP.get(language, language)
    instruction = (
        f"Classify the sentiment of the following {lang_name} sentence as Positive or Negative.\n\n"
        f"{FEW_SHOT}"
        f"Now classify this:\n"
        f"Sentence: {sentence}\n"
        f"Reply with exactly one word: Positive or Negative."
    )
    if label is not None:
        return (
            f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
            f"{RESPONSE_TEMPLATE}{label}<end_of_turn>"
        )
    else:
        return (
            f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
            f"{RESPONSE_TEMPLATE}"
        )

# Building the training dataset

* Applying `make_prompt` row-wise to build the full chat-formatted text for each of the 900 training examples.
* **Using ALL 900 samples** for training — no train/val split. With such a small dataset, holding out 10-20% for validation would starve the model of signal. Instead, evaluation is done by comparing baseline vs post-FT scores on the *same* stratified baseline sample (Section 5 vs Section 9).
* Converting the pandas DataFrame into a HuggingFace `Dataset` because `SFTTrainer` expects that format.
* Printing one example to visually confirm the chat template looks right before training.

In [7]:
t = time.time()

# Use ALL training data
train_df['text'] = train_df.apply(
    lambda r: make_prompt(r['sentence'], r['language'], r['label']), axis=1
)

train_dataset = Dataset.from_pandas(train_df[['text']].reset_index(drop=True))

print(f"Training on {len(train_dataset)} samples (all data)")
print("\nExample:")
print(train_dataset[0]['text'])

track_time(t, "Dataset prepared")

Training on 900 samples (all data)

Example:
<start_of_turn>user
Classify the sentiment of the following Punjabi sentence as Positive or Negative.

Examples:
- "வாடிக்கையாளர் சேவை மிகவும் நன்றாக இருந்தது" → Positive (Tamil)
- "এই পণ্যটি একদম বাজে" → Negative (Bengali)  
- "بہت اچھی سروس ہے" → Positive (Urdu)
- "ఈ సేవ చాలా చెత్తగా ఉంది" → Negative (Telugu)

Now classify this:
Sentence: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!
Reply with exactly one word: Positive or Negative.<end_of_turn>
<start_of_turn>model
Negative<end_of_turn>
Dataset prepared — 0.0s



# Tokenizer & Model loading

* Loading the Gemma 3-1B-IT tokenizer from the pre-downloaded Kaggle model mirror (no internet access is needed this way).
* `pad_token = eos_token` — Gemma does not ship with a dedicated padding token, so we reuse the EOS token for padding. This is a common pattern with causal-LM tokenizers.
* `padding_side = "right"` is set for **training** (the standard direction for causal LMs). It gets flipped to `"left"` later for generation, because during inference the model must continue from the rightmost token of the prompt.
* Decoding the `RESPONSE_TEMPLATE` token IDs is a sanity check to make sure the special Gemma chat tags are tokenized as single atomic tokens and not split into subwords — otherwise completion-only masking would not work correctly.

In [8]:
MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"  # right padding for training

print("Vocab size:", tokenizer.vocab_size)
print("Pad token: ", tokenizer.pad_token)

# Verify the response template tokenizes correctly
resp_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"Response template token ids: {resp_ids}")
print(f"Response template decoded: {tokenizer.decode(resp_ids)}")

Vocab size: 262144
Pad token:  <eos>
Response template token ids: [105, 4368, 107]
Response template decoded: <start_of_turn>model



# Loading Gemma 3-1B-IT in 4-bit (QLoRA setup)

* Loading the base model with **NF4 4-bit quantization** — the core technique behind QLoRA.
* `bnb_4bit_quant_type="nf4"` — NormalFloat-4, the quantization type tuned specifically for normally-distributed model weights. Gives better accuracy than standard int4.
* `bnb_4bit_compute_dtype=torch.float16` — the quantized weights are dequantized to fp16 *only* during the forward/backward pass to save compute time. Weights stay in 4-bit in memory.
* `bnb_4bit_use_double_quant=True` — also quantizes the quantization constants themselves. Saves roughly another 0.4 bits per parameter.
* `attn_implementation="eager"` — standard attention rather than flash-attention. Gemma 3 has specific requirements here and eager is the safest for this compute environment.
* `prepare_model_for_kbit_training` — wraps the model to enable gradient flow through the quantized layers and turns on gradient checkpointing for memory savings.
* Counting the 4-bit layers and printing GPU memory used (typically ~1.5 GB for a 1B model in NF4) as a sanity check.

In [9]:
t = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="cuda:0",
    attn_implementation="eager",
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

linear_layers = [(n, m) for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit)]
print(f"4-bit layers: {len(linear_layers)}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

track_time(t, "Model loaded")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

4-bit layers: 182
GPU memory used: 1.46 GB
Model loaded — 30.6s



# Pre-Fine-Tuning Evaluation (Baseline)

* Before attaching LoRA adapters, evaluating the **vanilla Gemma 3-1B-IT** model on a stratified sample from the training set.
* This gives a baseline Macro-F1 which is the number to beat after fine-tuning — without a baseline, any "improvement" claim from LoRA is ungrounded.
* `MAX_SEQ_LEN = 256` — kept short because:
  - Most sentences in the dataset are one-liners.
  - Shorter sequences = smaller batches in VRAM = faster inference.
  - Prompt + few-shot examples + answer fit comfortably under 256 tokens.

In [10]:
MAX_SEQ_LEN = 256

#  Shared inference helper

* One `predict_batch` function is used for **both** the baseline (pre-FT) evaluation and the post-FT evaluation — this guarantees any difference in scores comes from the model weights, not from inference-time code differences.
* `padding_side = "left"` is set here because during generation the prompt must end at the rightmost position — left-padding pushes the pad tokens to the start so all prompts in a batch align at the right edge.
* **Majority vote across 3 passes per sample** — this is the key robustness trick:
  - Pass 1: greedy decoding (`do_sample=False`) — deterministic, the model's most confident answer.
  - Passes 2 & 3: low-temperature sampling (`temperature=0.3`) — introduces a small amount of stochasticity to catch cases where greedy decoding is stuck in a wrong local choice.
  - Final prediction is the majority label across the 3 runs. This mimics self-consistency prompting and gave meaningful F1 gains over single-pass greedy.
* `max_new_tokens=4` — we only need "Positive" or "Negative", so capping at 4 tokens keeps generation fast.
* Decoding strips special tokens and checks for the substring `"positive"` — this is forgiving of minor format drift (extra spaces, punctuation, capitalization).
* Batch size 16 balances T4 VRAM (~15 GB available) and throughput.

In [11]:
#Shared inference helper (used by baseline AND post-FT evaluation)

tokenizer.padding_side = "left"  # left padding for generation
model.eval()

def predict_batch(sentences, languages, batch_size=16):
    results = []
    for i in range(0, len(sentences), batch_size):
        batch_s = sentences[i : i + batch_size]
        batch_l = languages[i : i + batch_size]
        prompts = [make_prompt(s, l) for s, l in zip(batch_s, batch_l)]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)

        votes = []
        # Run 3 passes: 1 greedy + 2 with low temperature
        for run_cfg in [
            {"do_sample": False},
            {"do_sample": True, "temperature": 0.3},
            {"do_sample": True, "temperature": 0.3},
        ]:
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=4,
                    pad_token_id=tokenizer.eos_token_id, **run_cfg
                )
            input_len = inputs["input_ids"].shape[1]
            run_preds = []
            for out in outputs:
                decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip().lower()
                run_preds.append("Positive" if "positive" in decoded else "Negative")
            votes.append(run_preds)

        # Majority vote across 3 runs per sample
        for j in range(len(batch_s)):
            sample_votes = [votes[r][j] for r in range(3)]
            results.append(max(set(sample_votes), key=sample_votes.count))

        if i % (batch_size * 3) == 0:
            print(f"  {min(i+batch_size, len(sentences))}/{len(sentences)}")

    return results

In [12]:
#  Build stratified baseline sample
baseline_df, _ = train_test_split(
    train_df, test_size=0.8, stratify=train_df['label'], random_state=42
)
# Cap at 200 samples to keep runtime reasonable on T4
baseline_df = baseline_df.sample(min(200, len(baseline_df)), random_state=42).reset_index(drop=True)

print(f'Baseline evaluation set: {len(baseline_df)} samples')
print(baseline_df['label'].value_counts().to_string())
print(baseline_df['language'].value_counts().to_string())

Baseline evaluation set: 180 samples
label
Positive    91
Negative    89
language
mr    18
ur    17
ta    17
gu    16
as    16
bd    15
or    13
bn    12
ml    12
kn    12
te    12
hi    11
pa     9


# Building the stratified baseline evaluation set

* Using `train_test_split` with `stratify=train_df['label']` to draw a sample that preserves the Positive/Negative class ratio — an unstratified random sample could end up heavily skewed and give a misleading baseline.
* Capping at **200 samples** to keep baseline inference under a few minutes on T4 — running full 900-sample inference just for a baseline would waste the GPU budget that's better spent on training.
* Printing per-label and per-language counts to confirm the baseline sample reasonably covers all 13 languages.

In [13]:
# Run baseline inference 
print('Running baseline (pre-fine-tuning) inference ...')
t = time.time()

baseline_preds = predict_batch(
    baseline_df['sentence'].tolist(),
    baseline_df['language'].tolist(),
    batch_size=16,
)

track_time(t, 'Baseline inference complete')

# Convert numeric labels to strings if needed 
true_labels = baseline_df['label'].tolist()
if isinstance(true_labels[0], (int, float)):
    inv_map = {1: 'Positive', 0: 'Negative'}
    true_labels = [inv_map[int(l)] for l in true_labels]

# Metrics
baseline_macro_f1 = f1_score(true_labels, baseline_preds, average='macro')
baseline_acc      = (np.array(true_labels) == np.array(baseline_preds)).mean()

print('  PRE-FINE-TUNING BASELINE RESULTS')
print(f'  Macro F1  : {baseline_macro_f1:.4f}')
print(f'  Accuracy  : {baseline_acc:.4f}')
print()
print(classification_report(true_labels, baseline_preds, target_names=['Negative', 'Positive']))

Running baseline (pre-fine-tuning) inference ...
  16/180
  64/180
  112/180
  160/180
Baseline inference complete — 31.1s

  PRE-FINE-TUNING BASELINE RESULTS
  Macro F1  : 0.8330
  Accuracy  : 0.8333

              precision    recall  f1-score   support

    Negative       0.80      0.89      0.84        89
    Positive       0.88      0.78      0.83        91

    accuracy                           0.83       180
   macro avg       0.84      0.83      0.83       180
weighted avg       0.84      0.83      0.83       180



# Per-language accuracy breakdown (baseline)

* Breaking down baseline accuracy by language to see which of the 13 Indic languages the vanilla Gemma model handles well out-of-the-box, and which ones need the most help from fine-tuning.
* Expected pattern: Hindi scores highest (Gemma 3 officially supports it), while low-resource languages like Bodo, Assamese, and Odia usually sit at the bottom.
* `baseline_results` dict persists the overall Macro-F1 and accuracy so the post-FT section can compute a clean delta.

In [14]:
# Per-language accuracy breakdown 
baseline_df = baseline_df.copy()
baseline_df['pred'] = baseline_preds
baseline_df['true_str'] = [inv_map[int(l)] if isinstance(l, (int, float)) else l
                            for l in baseline_df['label']]
baseline_df['correct'] = baseline_df['true_str'] == baseline_df['pred']

lang_acc = (
    baseline_df.groupby('language')['correct']
    .agg(['sum', 'count'])
    .assign(accuracy=lambda d: d['sum'] / d['count'])
    .rename(columns={'sum': 'correct', 'count': 'total'})
    .sort_values('accuracy', ascending=False)
)
lang_acc.index = lang_acc.index.map(lambda c: f"{LANG_MAP.get(c, c)} ({c})")
print('── Per-language accuracy (baseline) ──')
print(lang_acc.to_string())

# Persist results for post-FT comparison
baseline_results = {'macro_f1': baseline_macro_f1, 'accuracy': baseline_acc}
print(f'\nBaseline saved  =>  Macro F1 = {baseline_macro_f1:.4f}  |  Accuracy = {baseline_acc:.4f}')

── Per-language accuracy (baseline) ──
                correct  total  accuracy
language                                
Hindi (hi)           11     11  1.000000
Urdu (ur)            17     17  1.000000
Marathi (mr)         18     18  1.000000
Malayalam (ml)       11     12  0.916667
Telugu (te)          11     12  0.916667
Tamil (ta)           15     17  0.882353
Gujarati (gu)        14     16  0.875000
Kannada (kn)         10     12  0.833333
Punjabi (pa)          7      9  0.777778
Bengali (bn)          9     12  0.750000
Odia (or)             9     13  0.692308
Assamese (as)        11     16  0.687500
Bodo (bd)             7     15  0.466667

Baseline saved  =>  Macro F1 = 0.8330  |  Accuracy = 0.8333


# LoRA Configuration

* LoRA (Low-Rank Adaptation) adds small trainable "adapter" matrices alongside the frozen base weights instead of fine-tuning the full 1B parameters.
* Hyperparameter choices:
  - `r=32` — rank of the adapter matrices. Chose 32 (instead of the common 16) to give the adapters more expressiveness, since we're asking them to adapt the model to 13 languages at once. Memory cost is still negligible on a 1B model.
  - `lora_alpha=64` — scaling factor. Kept at **2× r** (the standard convention), which yields an effective scale of `alpha/r = 2`.
  - `lora_dropout=0.05` — light regularization since the dataset is small (900 samples) and overfitting is a real risk.
* `target_modules` — attaching LoRA to **all linear layers** in both attention (`q_proj, k_proj, v_proj, o_proj`) and the MLP (`gate_proj, up_proj, down_proj`). Targeting attention alone is common but gives less capacity; full-coverage LoRA was chosen because the task needs cross-lingual understanding, not just a classification head.
* `bias="none"` — biases are not adapted; standard LoRA setting.
* `task_type=TaskType.CAUSAL_LM` — tells PEFT this is autoregressive generation, not classification — the loss is next-token cross-entropy.
* The `get_peft_model` call is **commented out** because the trained adapter is loaded from disk in the next cell instead of training from scratch each run.

In [15]:
# r=32 gives more expressiveness than r=16 — still very memory efficient on a 1B model
# alpha=64 (2x r) maintains the standard scaling ratio
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

# Loading the saved adapter

* Reusing a previously trained LoRA adapter from a Kaggle dataset mirror — this avoids re-running the 8-epoch training loop on every submission attempt, saving ~30-40 minutes of compute per run.
* `is_trainable=False` — we're only doing inference with the adapter now, so freezing it is both faster and safer.
* `model.eval()` switches off dropout and sets the model to inference mode.

In [16]:
# Load saved adapter weights
model = PeftModel.from_pretrained(
    model,
    "/kaggle/input/datasets/rishabpanwar8876/gemma3-best-adapter",
    is_trainable=False
)
model.eval()
print("Saved adapter loaded.")

Saved adapter loaded.


## Clean previous outputs before training

In [17]:
# import shutil
# for folder in ["gemma3-sft", "gemma3-lora-adapter"]:
#     path = f"/kaggle/working/{folder}"
#     if os.path.exists(path):
#         shutil.rmtree(path)
# for f in ["submission.csv", "submissions.csv"]:
#     path = f"/kaggle/working/{f}"
#     if os.path.exists(path):
#         os.remove(path)
# print("Output directory clean.")

## 7. Train config

# Training configuration (SFT + LoRA)

* This cell is **commented out** because the final run loads a pre-trained adapter. Below is the configuration that was used to produce the adapter the first time.
* Configuration rationale:
  - **Effective batch size = 8 × 8 = 64** (per-device × gradient accumulation). Large enough to stabilize gradients on a 900-sample dataset, small enough to fit in T4 VRAM.
  - `learning_rate=2e-4` — the standard LoRA learning rate; much higher than full fine-tuning LRs (usually 1e-5) because we're only training a tiny fraction of parameters.
  - `lr_scheduler_type="cosine"` + 50 warmup steps — smooth LR ramp-up and decay prevents training instability in the first few steps.
  - `weight_decay=0.01` — mild regularization.
  - `optim="paged_adamw_8bit"` — 8-bit AdamW with paged memory. Further reduces optimizer state memory (critical on T4).
  - `num_train_epochs=8` — more epochs than usual, chosen because the dataset is small and each epoch passes over only 900 samples.
  - `fp16=False, bf16=False` — precision is handled internally by bitsandbytes/QLoRA; setting these explicitly would conflict.
  - `gradient_checkpointing=True` — recomputes activations during backward pass instead of storing them. Trades compute for memory.
  - `completion_only_loss=True` — **the key trick**: loss is computed only on the `Positive`/`Negative` response tokens, not on the prompt. This means gradient signal is focused entirely on the model learning to produce the correct label rather than memorising the prompt.
* `SFTTrainer` wraps the model, dataset, tokenizer, and config together into a trainable object.

In [18]:
# MAX_SEQ_LEN = 256

# sft_config = SFTConfig(
#     output_dir="/kaggle/working/gemma3-sft",

#     # Checkpointing
#     save_strategy="epoch",
#     save_total_limit=1,

#     # No eval dataset this time (using all data for training)
#     eval_strategy="no",

#     # Batch & Gradient
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=8,   # effective batch = 64

#     # Optimizer
#     learning_rate=2e-4,
#     lr_scheduler_type="cosine",
#     warmup_steps=50,
#     weight_decay=0.01,
#     optim="paged_adamw_8bit",

#     # Epochs: more epochs since dataset is small
#     num_train_epochs=8,

#     # Precision: off (4-bit handles it internally)
#     fp16=False,
#     bf16=False,

#     # Memory
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     dataloader_pin_memory=False,

#     # Sequence
#     max_length=MAX_SEQ_LEN,
#     packing=False,
#     dataset_text_field="text",

#     # KEY: compute loss only on completion tokens, not prompt
#     # This is what Unsloth does — masks the user turn so the model
#     # only receives gradient signal from predicting Positive/Negative
#     completion_only_loss=True,

#     # Logging
#     logging_steps=10,
#     report_to="none",
# )

# trainer = SFTTrainer(
#     model=model,
#     args=sft_config,
#     train_dataset=train_dataset,
#     processing_class=tokenizer,
# )

# print("Trainer ready.")

In [19]:
# t = time.time()
# trainer.train()
# track_time(t, "Training complete")

In [20]:
# model.save_pretrained("/kaggle/working/gemma3-lora-adapter")
# tokenizer.save_pretrained("/kaggle/working/gemma3-lora-adapter")
# print("Adapter saved.")

# Inference on the test set

* Running `predict_batch` on all 100 test samples using the fine-tuned model (base + loaded LoRA adapter).
* Each sample goes through 3 generation passes (greedy + 2 low-temperature samples) and the majority vote is taken — same logic as baseline, for consistency.
* `pd.Series(predictions).value_counts()` prints the Positive/Negative distribution of predictions as a sanity check — a heavily skewed distribution (e.g. 95 Positive / 5 Negative) would signal the model is biased.

In [21]:
t = time.time()
predictions = predict_batch(
    test_df['sentence'].tolist(),
    test_df['language'].tolist(),
)
track_time(t, "Inference complete")
print(pd.Series(predictions).value_counts())

  16/100
  64/100
  100/100
Inference complete — 22.3s

Positive    53
Negative    47
Name: count, dtype: int64


# Post-Fine-Tuning Evaluation (Comparison vs Baseline)

* Running the same `predict_batch` on the **same 200-sample baseline set** used in Section 5 — the only variable changed is the model (vanilla → fine-tuned).
* This apples-to-apples comparison is what justifies saying "fine-tuning helped". Comparing to a different sample or a different inference recipe would make the comparison meaningless.
* Printing Macro-F1, accuracy, and per-class precision/recall — the same metrics reported for the baseline.

In [22]:
print('Running post-fine-tuning evaluation on baseline sample ...')
t = time.time()

postft_preds = predict_batch(
    baseline_df['sentence'].tolist(),
    baseline_df['language'].tolist(),
    batch_size=16,
)

track_time(t, 'Post-FT evaluation complete')

postft_macro_f1 = f1_score(true_labels, postft_preds, average='macro')
postft_acc      = (np.array(true_labels) == np.array(postft_preds)).mean()

print('  POST-FINE-TUNING RESULTS')
print(f'  Macro F1  : {postft_macro_f1:.4f}')
print(f'  Accuracy  : {postft_acc:.4f}')
print()
print(classification_report(true_labels, postft_preds, target_names=['Negative', 'Positive']))

Running post-fine-tuning evaluation on baseline sample ...
  16/180
  64/180
  112/180
  160/180
Post-FT evaluation complete — 39.7s

  POST-FINE-TUNING RESULTS
  Macro F1  : 0.9444
  Accuracy  : 0.9444

              precision    recall  f1-score   support

    Negative       0.95      0.93      0.94        89
    Positive       0.94      0.96      0.95        91

    accuracy                           0.94       180
   macro avg       0.94      0.94      0.94       180
weighted avg       0.94      0.94      0.94       180



# Fine-tuning impact summary

* Building a small DataFrame showing Baseline → After-FT for both Macro-F1 and accuracy, plus the delta.
* The delta (`+`) is the clearest single number justifying the fine-tuning effort. If `delta_f1` is negative or negligible, the LoRA config / training setup needs revisiting.

In [23]:
# Side-by-side comparison 
delta_f1  = postft_macro_f1 - baseline_results['macro_f1']
delta_acc = postft_acc      - baseline_results['accuracy']

comparison = pd.DataFrame({
    'Metric'   : ['Macro F1', 'Accuracy'],
    'Baseline' : [f"{baseline_results['macro_f1']:.4f}", f"{baseline_results['accuracy']:.4f}"],
    'After FT' : [f'{postft_macro_f1:.4f}', f'{postft_acc:.4f}'],
    'Delta'    : [f'{delta_f1:+.4f}', f'{delta_acc:+.4f}'],
})

print('  FINE-TUNING IMPACT SUMMARY')
print(comparison.to_string(index=False))

  FINE-TUNING IMPACT SUMMARY
  Metric Baseline After FT   Delta
Macro F1   0.8330   0.9444 +0.1114
Accuracy   0.8333   0.9444 +0.1111


# Building the submission file

* Mapping the string predictions (`"Positive"` / `"Negative"`) back to integer labels (`1` / `0`) to match the competition's expected submission format.
* Building a two-column DataFrame (`ID`, `label`) aligned to `test_df['ID']` to preserve row order — mis-ordering here would silently destroy the F1 score.
* Saving to `/kaggle/working/submission.csv`, which is the path Kaggle's evaluator expects for scoring.

In [24]:
label_map = {"Positive": 1, "Negative": 0}

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'label': [label_map[p] for p in predictions]
})
submission.to_csv("/kaggle/working/submission.csv", index=False)